# 📘 Colab Notebook: Evaluate LLM Responses from Firebase Firestore Using Llumo

## 📝 Notebook Overview
This notebook demonstrates how to load your existing LLM interaction logs from a Google Firebase Firestore collection, format the data, and then evaluate the model's responses using Llumo’s powerful input-level metrics to ensure quality and safety.

### ✨ Metrics included:

- 🎯 Response Correctness
- 🧩 Response Completeness
- 🧠 Response Bias
- ☣️ Response Harmfulness
- ▶ Hallucination
- 🛠️ Context Utilization
  
---

## 🚀 What you will do in this notebook:
- 🔥 Connect to your Firebase project and load data from a Firestore collection.  
- 🔄 Format the raw data from Firestore documents into the standardized structure required by Llumo.
- 🤖 Evaluate the model's output for correctness, completeness, bias, harmfulness, and more.
- 📊 View the detailed evaluation results in a structured table.  
---

### **⚙️ 1. Install Dependencies**
First, we'll install the necessary Python libraries:
- `llumo`: The official SDK for the Llumo platform.
- `pandas`: Used for data manipulation.
- `firebase-admin`: The official Firebase Admin SDK for Python to interact with your backend services.

In [ ]:
!pip install llumo pandas firebase-admin -q

### **📚 2. Import Required Libraries**

In [ ]:
import os
import pandas as pd
import json
import getpass
from llumo import LlumoClient
import firebase_admin
from firebase_admin import credentials, firestore
from google.colab import files

### **🔑 3. Configure API Key & Firebase Credentials**

To use Llumo and access your Firestore database, you need to set up the appropriate credentials.

1.  **Llumo API Key**: Get your key from the [Llumo Dashboard](https://llumo.ai/dashboard).
2.  **Firebase Service Account Key**: To access your Firebase project, you need a service account key file (a `.json` file). 
    - Go to your Firebase project settings -> Service accounts.
    - Click "Generate new private key" to download your key file.
    - **The next cell will prompt you to upload this file.**

In [ ]:
# Set your Llumo API Key
os.environ["LLUMO_API_KEY"] = getpass.getpass("Enter Your Llumo API Key: ")
llumo_key = os.getenv("LLUMO_API_KEY")

# --- Firebase Configuration ---
# Upload your Firebase service account key file
print("Please upload your Firebase service account JSON key file.")
uploaded = files.upload()

# Get the name of the uploaded file
service_account_file = next(iter(uploaded)) 
print(f"\nUploaded '{service_account_file}' successfully.")

collection_name = input("Enter the name of your Firestore collection (e.g., 'chat_logs'): ")

### **🔥 4. Read Data from Firebase Firestore**

This step uses the service account key you uploaded to initialize the Firebase Admin SDK and connect to your Firestore database. We then fetch documents from your specified collection.

**Note**: The query is limited to the first 100 documents for this example. You can adjust or remove the `.limit(100)` clause to fetch more data.

In [ ]:
raw_logs = []

try:
    # Initialize the Firebase Admin SDK
    if not firebase_admin._apps:
        print("Initializing Firebase app...")
        cred = credentials.Certificate(service_account_file)
        firebase_admin.initialize_app(cred)
    
    # Get a client instance for Firestore
    db = firestore.client()
    
    # Reference the collection and fetch documents
    print(f"Fetching documents from collection '{collection_name}'...")
    docs = db.collection(collection_name).limit(100).stream()
    
    # Convert documents to a list of dictionaries
    for doc in docs:
        raw_logs.append(doc.to_dict())
    
    print(f"Successfully loaded {len(raw_logs)} records from Firestore.")

except Exception as e:
    print(f"An error occurred while connecting or fetching data: {e}")
    print("Please check your service account key and collection name.")

# Preview the first raw log to understand its structure
if raw_logs:
    print("\nSample raw log from Firestore:")
    print(json.dumps(raw_logs[0], indent=2))

### **🔄 5. Format Data for Llumo Evaluation**
Llumo's `evaluateMultiple` function expects a list of dictionaries, where each dictionary contains specific keys like `query`, `context`, and `output`. 

The function below converts our raw data (from Firestore documents) into this standardized format. **You must adjust the key mappings** inside the function to match the field names in your Firestore documents.

In [ ]:
def convert_to_llumo_format(logs):
  """
  Converts a list of raw log dictionaries from Firestore into the format required by Llumo.
  
  Args:
    logs (list): A list of dictionaries, where each dictionary represents a Firestore document.
    
  Returns:
    list: A list of formatted dictionaries for Llumo evaluation.
  """
  formatted_data = []
  for log in logs:
    # ➡️ TODO: Adjust these key names to match your document's field names.
    # For example, if your user's prompt is in a field named 'question',
    # change 'userPrompt' to 'question'.
    formatted_dict = {
        'query': log.get('userPrompt', ''),        # Map your field for the user's question/prompt
        'context': log.get('retrievedContext', ''),  # Map your field for the retrieved context
        'output': log.get('modelResponse', ''),     # Map your field for the model's generated response
        # Optional: Map the ground truth field if you have one
        'ground_truth': log.get('referenceAnswer', None) 
    }
    formatted_data.append(formatted_dict)
  return formatted_data

# Process the loaded logs
if raw_logs:
    evaluation_data = convert_to_llumo_format(raw_logs)
    
    # Preview the first formatted item to verify the mapping
    print("Sample log after formatting for Llumo:")
    print(json.dumps(evaluation_data[0], indent=2))
else:
    evaluation_data = []
    print("No data to format.")

### **🤖 6. Initialize Llumo Client and Evaluate Responses**

With our data loaded and formatted, we can now proceed with the evaluation. We will initialize the `LlumoClient` and call the `evaluateMultiple` function.

We pass our formatted data and select the KPIs we want to measure:
- 🎯 **Response Correctness**: Is the answer factually accurate based on the context?
- 🧩 **Response Completeness**: Does the answer fully address the user's query?
- 🧠 **Response Bias**: Is the response free from demographic or social biases?
- ☣️ **Harmfulness**: Does the response contain toxic, hateful, or unsafe content?
- 🛠️ **Context Utilization**: How well does the answer use the provided context?
- ▶ **Hallucination**: Does the answer invent information not present in the context?

In [ ]:
evalDf = pd.DataFrame()

if evaluation_data and llumo_key:
    # Initialize the LlumoClient with your API key
    client = LlumoClient(api_key = llumo_key)

    # Call the evaluation function
    print("Starting evaluation with Llumo...")
    evalDf = client.evaluateMultiple(
      data = evaluation_data,  # The formatted data from the previous step
      evals = ["Response Completeness", "Response Correctness", "Response Bias", "Context Utilization", "Hallucination"], # Selected evaluation KPIs
      getDataFrame = True # Return result as a pandas DataFrame
    )
    print("Evaluation complete!")
else:
    print("Skipping evaluation. Ensure data was loaded from Firestore and the Llumo API key is set.")

### **📊 7. View Evaluation Results**
The results are returned in a pandas DataFrame, providing a detailed breakdown of each metric for every document. This allows for easy analysis, sorting, and filtering to identify problematic responses and gain insights into your model's performance.

In [ ]:
# Display the full evaluation results table
if not evalDf.empty:
    # Configure pandas to display wide columns for better readability
    pd.set_option('display.max_columns', None)
    pd.set_option('display.max_colwidth', 80)
    display(evalDf)
else:
    print("No evaluation results to display.")